In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
f = np.load("longtrack_unipolar.npz")
label = "tpc0_batch10"
# f = np.load("angle3_pid13.npz")
# label = "tpc5_batch0"
# f = np.load("single_track_for_sp_unipolar_pid13_angle00.npz")
response = np.load("/home/yousen/Public/ndlar_shared/data/unipolar_response_v2a_distance_10p431cm_binsize_0p04434cm_tick0p05us.npy")

# f = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250609/single_segment_for_larndsim_unipolar_center.npz')
# f = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_tred_comp20250615/single_segment_for_larndsim_unipolar_center_30cm.npz')
# response = np.load("/home/yousen/Public/ndlar_shared/data/responses/response_v2a_center_unipolar_30p431cm_0p050us_bin_size0p04434.npz")["response"]


In [ ]:
f.files, f['response_path']

In [ ]:
label = 'tpc0_batch10'
def plot_conv(response_index = (4, 4), label = 'tpc0_batch10'):
    
    plt.close('all')
    upix, inv = np.unique(f[f'effq_{label}_location'][:,:2], axis=0, return_inverse=True)

    for ip, pix in enumerate(upix):

        print(ip)
        if ip >= 5:
            break
        
        mpix = ip == inv
        q = f[f'effq_{label}'][mpix]
        t = f[f'effq_{label}_location'][:,-1][mpix]
        q = q[:,-1]
    
        if np.sum(q) < 5E3:
            continue


        fig, ax = plt.subplots()
        ax.plot(t, q)
        ax.set_title(f'projected effq; pix index ({upix[ip][0]}, {upix[ip][1]})')

        fig2, ax2 = plt.subplots()
        mc = f[f'current_{label}_location'][:,:2] == pix
        mc = np.all(mc, axis=1)
        qc = np.squeeze(f[f'current_{label}'][mc]) * 1E3
        tc = f[f'current_{label}_location'][mc][:,-1] + np.arange(qc.shape[0])
        ax2.plot(tc, qc, label= 'induced current (tred)')
        ax2.set_xlim(ax.get_xlim()[0] - 100, ax.get_xlim()[1]+50)

        tpeak = tc[np.argmax(qc)]

        tres = np.arange(response.shape[-1]) + tpeak - np.argmax(response[response_index])
        ax2.plot(tres, response[response_index]/np.max(response[response_index]) * np.max(qc), label=f'response[{str(response_index)}] (scaled on q & shifted)')
    
        qconv = np.convolve(q, response[response_index])*0.05
        tconv = np.arange(qconv.shape[0]) + tpeak - np.argmax(qconv)
    
        ax2.plot(tconv, qconv, label='conv(proj. q, response) (shifted on t)')
        ax2.plot(tconv, qconv / np.max(qconv) * np.max(qc), label=f'conv(proj. q, response[{str(response_index)}]) (scaled on q & shifted on t)')
        ax2.set_title(f'induced current; pix index ({upix[ip][0]}, {upix[ip][1]})')
        ax2.vlines([t[0], t[-1]], *ax2.get_ylim(), label='q range', linestyle='-.')

        ax2.legend()

In [ ]:
for i in range(5):
    for j in range(i,5):
        print(i, j, np.any(np.abs(response[0,0] - response[i,j]) > 1E-5), )

        plt.plot(response[i,j], label=f'response[{i}, {j}]')
plt.xlim(1200, 1350)
plt.legend()
plt.title('field response')

In [ ]:
plot_conv((0,0))

In [ ]:
plot_conv((4,4), label)

In [ ]:
def plot_q(label = 'tpc0_batch10'):
    
    plt.close('all')
    upix, inv = np.unique(f[f'effq_{label}_location'][:,:2], axis=0, return_inverse=True)

    for ip, pix in enumerate(upix):

        mpix = ip == inv
        q = f[f'effq_{label}'][mpix]
        t = f[f'effq_{label}_location'][:,-1][mpix]
        q = q[:,-1]
    
        if np.sum(q) < 5E3:
            continue

        fig, ax = plt.subplots()
        ax.plot(t, q)
        ax.set_title(f'projected effq; pix index ({upix[ip][0]}, {upix[ip][1]})')

In [ ]:
plot_q(label)

In [ ]:
h = f[f'hits_{label}']
print(h)
dh = h[0,:3] - h[-1,:3]
print(dh[0]/dh[2])
np.arctan(dh[0]/dh[2])
np.rad2deg(np.arctan(dh[0]/dh[2])
          )